# Menlo Robot SDK — Simple Local Demo

This notebook demonstrates the smallest useful Menlo SDK workflow:

1. Load `MENLO_API_KEY` from `.env`
2. Create a simulated robot
3. Connect to it
4. Open the browser-based 3D simulator
5. Discover and invoke a movement skill
6. Read robot state and capture its camera
7. Disconnect and delete the robot

> **Important:** Keep the 3D viewer open and visible while running commands. The robot simulation executes inside that browser tab.

## 1. Install dependencies

For local use, first create the isolated Python 3.12 environment described in `README.md`, then select that kernel in Jupyter. Run this cell once for the environment and restart the kernel if packages change. Isolation avoids dependency conflicts with system Python.

In [ ]:
%pip install -q -r requirements.txt

## 2. Load local configuration

The `.env` file should be in the same directory as this notebook and contain:

```text
MENLO_API_KEY=your_key_here
```

The key is loaded into memory but never printed.

In [ ]:
import asyncio
import os
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

MENLO_API_KEY = os.getenv("MENLO_API_KEY")
RCS_URL = "https://api.menlo.ai/rcs"
VIEWER_BASE_URL = "https://sim.menlo.ai"

if not MENLO_API_KEY:
    raise RuntimeError("MENLO_API_KEY is missing from .env")

print("Configuration loaded. API key is set.")

## 3. Create a simulated robot

`AsyncClient` manages resources on the Menlo platform. Creating a robot returns its unique ID. The variables are initialized first so the cleanup cell remains safe even if setup fails partway through.

In [ ]:
from menlo_robot_sdk import AsyncClient, connect

client = None
session = None
robot_id = None

client = AsyncClient(rcs_url=RCS_URL, api_key=MENLO_API_KEY)
created = await client.robots.create(
    name=f"simple-demo-{int(time.time())}",
    model="asimov-v0",
)
robot_id = created.robot.id
print(f"Created robot: {robot_id}")

## 4. Connect to the robot

The returned `session` is the main SDK object. Use it to discover skills, invoke actions, read state, and retrieve camera images.

In [ ]:
session = await connect(
    client,
    robot_id,
    worker_names=[],
    # The browser simulator joins the room with a `simplesim` identity.
    rcw_identity_prefix="simplesim",
    join_livekit=True,
)
print(f"Connected to room: {session.session.room_name}")

## 5. Start the simulator

Run this cell, click the displayed link, and leave that browser tab open and visible. Wait until the warehouse and robot have loaded before running the next cell.

In [ ]:
from IPython.display import HTML, display
from menlo_robot_sdk.experimental import generate_room_key

room_key = await generate_room_key(client, robot_id)
viewer_url = f"{VIEWER_BASE_URL}/?key={room_key}"
display(HTML(f'<a href="{viewer_url}" target="_blank" rel="noopener"><b>Open the Menlo 3D viewer</b></a>'))

## 6. Discover the robot's skills

Skills are supplied by the simulator runtime. This polling loop waits for the viewer to join, then prints each action exposed by the robot.

In [ ]:
async def wait_for_skills(timeout_s=120):
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        try:
            skills = await session.discover_skills()
            if skills:
                return skills
        except (RuntimeError, TimeoutError):
            pass
        await asyncio.sleep(2)
    raise TimeoutError("No skills found. Is the 3D viewer open and fully loaded?")

skills = await wait_for_skills()
print("Available skills:")
for skill in skills:
    print(f"- {skill.name}: {skill.description}")

## 7. Read the robot's state

State reads are observations; they do not move the robot. Here we inspect its status and pose before issuing a command.

In [ ]:
async def show_robot_state(label):
    state = await session.state.get("robot_status")
    pose = state.robot.pose
    print(f"{label}: status={state.robot.status}")
    if pose:
        x, y, z = pose.position
        print(f"  position=({x:+.2f}, {y:+.2f}, {z:+.2f}), yaw={pose.yaw_deg:+.1f}°")
    return state

before = await show_robot_state("Before movement")

## 8. Invoke one movement skill

`session.invoke` calls a named skill with a dictionary of arguments. This command walks forward slowly for one second and then stops.

In [ ]:
result = await session.invoke(
    "set_velocity",
    {"vx": 0.4, "vy": 0.0, "wz": 0.0, "duration_s": 1.0},
    timeout_s=60,
)
print(f"Movement result: {result.status}")
after = await show_robot_state("After movement")

## 9. Capture the robot's point-of-view camera

`get_vision("pov")` returns JPEG bytes from the camera mounted on the robot.

In [ ]:
from IPython.display import Image

jpeg = await session.get_vision("pov")
print(f"Received {len(jpeg):,} bytes from the POV camera")
display(Image(data=jpeg, format="jpeg"))

## 10. Clean up

Always run this cell when finished. It disconnects the real-time session, deletes the temporary simulated robot, and closes the API client.

In [ ]:
if session is not None:
    await session.disconnect()
    session = None
    print("Session disconnected.")

if client is not None:
    if robot_id is not None:
        await client.robots.delete(robot_id)
        robot_id = None
        print("Robot deleted.")
    await client.aclose()
    client = None
    print("Client closed.")

## What to try next

Once this works, make one small change at a time:

- Change `vx` to adjust forward/backward movement.
- Use `vy` to sidestep.
- Combine a small `vx` with `wz` to turn; this humanoid cannot reliably spin in place.
- Inspect a skill's printed description before choosing its arguments.
- Read `robot_status` after every action instead of assuming it succeeded.